In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Code to generate variable attribution dataset

In [ ]:
import random
import itertools
from typing import Callable, List, Dict, Any, Set

def generate_batch_logic_gate_data(
    num_samples: int,               # Batch size: number of datasets to generate this run
    num_vars: int,
    num_trials: int,
    num_active_vars: int,
    logic_func: Callable[..., int],
    logic_description: str,
    start_seed: int = 0,            # <--- KEY FEATURE: The "Page Number" (Offset)
    max_retries: int = 5000
) -> List[Dict[str, Any]]:
    """
    Generates a batch of logic puzzle datasets.
    Crucially, it uses (start_seed + i) to seed the random number generator for each sample.
    This ensures that sample #50 is always identical, whether generated in a batch of 100
    or in a second batch starting at index 50.
    """

    dataset_batch = []
    all_vars = list(range(num_vars))

    # 1. Pre-calculate Truth Table Structure
    # This is the base combination that active vars must cover
    full_truth_table = list(itertools.product([0, 1], repeat=num_active_vars))
    num_required_rows = len(full_truth_table)
    all_permutations = list(itertools.permutations(all_vars, num_active_vars))

    if num_trials < num_required_rows:
        raise ValueError(f"num_trials ({num_trials}) is too small to cover the truth table.")

    # --- Batch Generation ---
    for i in range(num_samples):
        # =================================================================
        # CORE MECHANISM: The "Bookmark" Logic
        # =================================================================
        # Calculate the absolute ID (the "page number") of this sample
        current_sample_id = start_seed + i

        # Force reset the random seed for this specific sample.
        # This guarantees that Sample #50 is mathematically determined by the integer 50.
        # It creates a deterministic mapping: Integer ID -> Unique Dataset.
        random.seed(current_sample_id)
        # =================================================================

        success = False
        attempts = 0

        while attempts < max_retries:
            attempts += 1

            # --- Step A: Determine Ground Truth (GT) Variables ---
            target_indices = random.sample(all_vars, num_active_vars)
            target_set = set(target_indices)

            # Initialize matrix (Trials x Vars) with zeros
            inputs = [[0] * num_vars for _ in range(num_trials)]

            # --- Step B: Fill GT Variables (Force Row-level Coverage) ---
            # 1. Ensure the first N rows completely cover the truth table
            for r in range(num_required_rows):
                vals = full_truth_table[r]
                for local_idx, var_idx in enumerate(target_indices):
                    inputs[r][var_idx] = vals[local_idx]

            # 2. Randomly fill GT variables for the remaining rows
            for r in range(num_required_rows, num_trials):
                for var_idx in target_indices:
                    inputs[r][var_idx] = random.randint(0, 1)

            # --- Step C: Fill Distractor Variables (Eliminate Distribution Shortcuts) ---
            # We mimic the GT distribution rather than using pure random.randint
            # to prevent the model from guessing based on simple frequency stats.

            # Calculate base counts from the truth table part
            base_zeros = num_required_rows // 2
            base_ones = num_required_rows - base_zeros

            other_indices = [v for v in all_vars if v not in target_indices]

            for var_idx in other_indices:
                col_data = []
                # 1. Ensure base diversity (mimic truth table part)
                col_data.extend([0] * base_zeros)
                col_data.extend([1] * base_ones)

                # 2. Pad to remaining length (mimic random part)
                remaining_len = num_trials - len(col_data)
                for _ in range(remaining_len):
                    col_data.append(random.randint(0, 1))

                # 3. Shuffle this column (Independent Identical Distribution)
                # This ensures the column looks balanced but has no correlation with GT.
                random.shuffle(col_data)

                # 4. Fill into the matrix
                for r in range(num_trials):
                    inputs[r][var_idx] = col_data[r]

            # --- Step D: Shuffle Row Order ---
            # Essential: otherwise the first N rows are obviously the truth table.
            random.shuffle(inputs)

            # --- Step E: Calculate Outcome ---
            outcomes = []
            try:
                for trial in inputs:
                    active_vals = [trial[idx] for idx in target_indices]
                    res = int(logic_func(*active_vals))
                    outcomes.append(res)
            except Exception:
                continue

            # Ensure non-trivial outcome (must have at least one 0 and one 1)
            if len(set(outcomes)) < 2:
                continue

            # --- Step F: Uniqueness Verification (Ambiguity Check) ---
            # Ensure no OTHER combination of variables accidentally explains the results.

            is_ambiguous = False

            for cand in all_permutations:
                # Optimization: Check if this candidate set explains the data
                cand_is_valid = True
                for r_idx, row in enumerate(inputs):
                    vals = [row[v] for v in cand]
                    if int(logic_func(*vals)) != outcomes[r_idx]:
                        cand_is_valid = False
                        break # Fail early if any row doesn't match

                if cand_is_valid:
                    # If a valid solution is found, it MUST be the Ground Truth set
                    if set(cand) != target_set:
                        is_ambiguous = True
                        break # Found an ambiguity, stop checking

            if is_ambiguous:
                # Re-generate. Ambiguity is rare but possible.
                continue

            # --- Step G: Success - Package Data ---
            prompt_text = (
                "You are a scientist analyzing experimental records.\n"
                f"We have {num_vars} variables (V0 - V{num_vars-1}). We want to identify the specific variables that dictate the outcome.\n"
                f"HYPOTHESIS: The outcome is determined by a hidden function ({logic_description}) acting on {num_active_vars} variables.\n"
                "Note: There is only one unique solution set.\n"
                "Experimental Log:\n"
            )
            for r_idx, (trial, out) in enumerate(zip(inputs, outcomes)):
                vars_str = ", ".join([f"V{v}={val}" for v, val in enumerate(trial)])
                prompt_text += f"Trial {r_idx+1}: {{ {vars_str} }} -> Outcome: {out}\n"
            prompt_text += f"\nBased on the data, identify the unique {num_active_vars} variables that cause the outcome. End with 'Final Answer:'"

            dataset_batch.append({
                "id": current_sample_id, # Store ID for tracking
                "full_prompt": prompt_text,
                "inputs": inputs,
                "outcomes": outcomes,
                "ground_truth": target_indices
            })
            success = True
            break # Successfully generated a sample, break retry loop

        if not success:
            print(f"Warning: Failed to generate sample ID {current_sample_id} after {max_retries} retries.")

    return dataset_batch


def verify_dataset(
    data: Dict,
    logic_func: Callable,
    num_vars: int,
    num_active_vars: int
) -> bool:
    """
    Standard validation function to check if a generated dataset is valid
    (Correct GT and Unique Solution).
    """
    try:
        inputs = data['inputs']
        outcomes = data['outcomes']
        gt_indices = data['ground_truth']
        gt_set = set(gt_indices)

        # 1. Check GT correctness
        for i, row in enumerate(inputs):
            vals = [row[idx] for idx in gt_indices]
            if int(logic_func(*vals)) != outcomes[i]:
                return False

        # 2. Check Uniqueness (No other set works)
        all_perms = list(itertools.permutations(range(num_vars), num_active_vars))

        for cand in all_perms:
            match = True
            for i, row in enumerate(inputs):
                vals = [row[idx] for idx in cand]
                if int(logic_func(*vals)) != outcomes[i]:
                    match = False
                    break

            if match:
                if set(cand) != gt_set:
                    return False # Ambiguity found
        return True

    except Exception:
        return False


In [ ]:
import math
import itertools
import pickle
import os
import re
from typing import Callable, List, Dict, Any

def get_theoretical_min_trials(n_vars: int, n_gt: int) -> int:
    """
    To compute the minimum number of trials, the following conditions must be met simultaneously:
    1. Sufficient to distinguish all combinations of variables (Information Theory)
    2. Sufficient to cover the truth table (Structural Constraint)
    """
    # 1. Information Lower Bound: The Amount of Information Required to "Distinguish Between Suspects"
    combinations = math.comb(n_vars, n_gt)
    info_min = math.ceil(math.log2(combinations)) if combinations > 0 else 0

    # 2. Lower bound of the structure: The number of rows required to fully display the logic (2^K)
    struct_min = 2 ** n_gt

    return max(info_min, struct_min)

def run_adaptive_experiment_pipeline(
    vars_list: List[int],
    trials_delta_list: List[int], # offset from the min trials
    gt_vars_num: int,
    samples_per_setting: int,
    logic_func: Callable,
    logic_description: str,
    save_dir: str,
    base_start_seed: int = 0
):
    """
    The required number of trials is automatically calculated based on the number of Vars.
    `trials_delta_list: [0, 1, 2]` indicates that trials of size [Min, Min+1, Min+2] are run respectively.
    """

    safe_desc = re.sub(r'[^a-zA-Z0-9()]', '_', logic_description)
    safe_desc = re.sub(r'_+', '_', safe_desc).strip('_')
    filename = f"{gt_vars_num}_vars_adaptive_sample_{samples_per_setting}_{safe_desc}.pkl"

    print(f"=== Adaptive Experiment Start ===")
    print(f"Target: {filename}")
    print(f"Vars: {vars_list}")
    print(f"Deltas: {trials_delta_list} (Offset from theoretical minimum)")
    print("="*40)

    dataset = []

    configs = []
    for n_vars in vars_list:
        min_t = get_theoretical_min_trials(n_vars, gt_vars_num)
        for delta in trials_delta_list:
            configs.append((n_vars, min_t + delta))

    total_configs = len(configs)

    for idx, (n_vars, n_trials) in enumerate(configs):

        # Sanity Check
        possible_combos = math.comb(n_vars, gt_vars_num)
        capacity = 2 ** n_trials

        ratio = capacity / possible_combos if possible_combos > 0 else 0
        print(f"[{idx+1}/{total_configs}] V={n_vars}, T={n_trials} | Combos={possible_combos}, Cap={capacity} (Ratio={ratio:.2f})")

        # If Ratio < 1, it means that even with theoretical calculations, the integer rounding might still be insufficient
        if capacity < possible_combos:
             print(f"   -> SKIP (Mathematically impossible)")
             continue

        try:
            batch_data = generate_batch_logic_gate_data(
                num_samples=samples_per_setting,
                num_vars=n_vars,
                num_trials=n_trials,
                num_active_vars=gt_vars_num,
                logic_func=logic_func,
                logic_description=logic_description,
                start_seed=base_start_seed,
                max_retries=50000
            )
        except Exception as e:
            print(f"   -> [Error] {e}")
            batch_data = []

        if not batch_data:
            print(f"   -> ⚠️ FAILED (Hard generation)")
        else:
            # Metadata
            for sample in batch_data:
                sample['num_trials'] = n_trials
                sample['trials_delta'] = n_trials - get_theoretical_min_trials(n_vars, gt_vars_num)
                sample["num_vars"] = n_vars
                sample["num_gt_var"] = gt_vars_num
                sample["difficulty_ratio"] = ratio
                dataset.append(sample)
            print(f"   -> ✅ OK")

    # Save
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    full_path = os.path.join(save_dir, filename)
    with open(full_path, 'wb') as f:
        pickle.dump(dataset, f)
    print(f"\nSaved {len(dataset)} items to: {full_path}")

In [ ]:
path_to_save = '/content/drive/MyDrive/inference-time-scale-adaptation/dataset/'

os.makedirs(path_to_save, exist_ok=True)

logic_functions = {
    # === Group k=1: AND-like ===
    "(NOT A) AND (NOT B)": lambda a, b: 1 if (not a and not b) else 0,
    "A AND (NOT B)": lambda a, b: 1 if (a and not b) else 0,
    "(NOT A) AND B": lambda a, b: 1 if (not a and b) else 0,
    "A AND B": lambda a, b: 1 if (a and b) else 0,

    # === Group k=3: OR-like ===
    "NOT (A AND B)": lambda a, b: 0 if (a and b) else 1,
    "(NOT A) OR B": lambda a, b: 0 if (a and not b) else 1,
    "A OR (NOT B)": lambda a, b: 0 if (not a and b) else 1,
    "A OR B": lambda a, b: 1 if (a or b) else 0,

    # === Group k=2: XOR/XNOR ===
    "A XOR B": lambda a, b: 1 if (a != b) else 0,
    "A XNOR B": lambda a, b: 1 if (a == b) else 0,
}

for desc, func in logic_functions.items():
    print(f"🚀 Running experiment for: {desc}")
    run_adaptive_experiment_pipeline(
        vars_list=[3,4,5,6,7,8,10,12,14,16],
        trials_delta_list=[0,1,2,3,4,5],
        gt_vars_num=2,
        samples_per_setting=5,
        logic_func=func,
        logic_description=desc,
        save_dir=path_to_save
    )

print("✅ All experiments finished!")

🚀 Running experiment for: (NOT A) AND (NOT B)
=== Adaptive Experiment Start ===
Target: 2_vars_adaptive_sample_5_NOT_A_AND_NOT_B.pkl
Vars: [3, 4, 5, 6, 7, 8, 10, 12, 14, 16]
Deltas: [0, 1, 2, 3, 4, 5] (Offset from theoretical minimum)
[1/60] V=3, T=4 | Combos=3, Cap=16 (Ratio=5.33)
   -> ✅ OK
[2/60] V=3, T=5 | Combos=3, Cap=32 (Ratio=10.67)
   -> ✅ OK
[3/60] V=3, T=6 | Combos=3, Cap=64 (Ratio=21.33)
   -> ✅ OK
[4/60] V=3, T=7 | Combos=3, Cap=128 (Ratio=42.67)
   -> ✅ OK
[5/60] V=3, T=8 | Combos=3, Cap=256 (Ratio=85.33)
   -> ✅ OK
[6/60] V=3, T=9 | Combos=3, Cap=512 (Ratio=170.67)
   -> ✅ OK
[7/60] V=4, T=4 | Combos=6, Cap=16 (Ratio=2.67)
   -> ✅ OK
[8/60] V=4, T=5 | Combos=6, Cap=32 (Ratio=5.33)
   -> ✅ OK
[9/60] V=4, T=6 | Combos=6, Cap=64 (Ratio=10.67)
   -> ✅ OK
[10/60] V=4, T=7 | Combos=6, Cap=128 (Ratio=21.33)
   -> ✅ OK
[11/60] V=4, T=8 | Combos=6, Cap=256 (Ratio=42.67)
   -> ✅ OK
[12/60] V=4, T=9 | Combos=6, Cap=512 (Ratio=85.33)
   -> ✅ OK
[13/60] V=5, T=4 | Combos=10, Cap=16 (